In [ ]:
#Profile All Identity Cleaning
!pip install openpyxl


import pandas as pd
import re
from google.colab import files

# 1️⃣ Upload file
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 2️⃣ Read file
if file_name.endswith(".csv"):
    df = pd.read_csv(file_name)
else:
    df = pd.read_excel(file_name)

# 🔹 Column name exactly as in your sheet
raw_column_name = "profile.all_identities"

def clean_phone(phone):
    # Remove everything except digits
    phone = re.sub(r'[^0-9]', '', phone)

    # Remove leading 91 if present and length > 10
    if phone.startswith("91") and len(phone) > 10:
        phone = phone[-10:]

    # Keep only valid 10-digit numbers
    if len(phone) == 10:
        return phone
    else:
        return None

def extract_details(text):
    if pd.isna(text):
        return pd.Series(["", "", "", ""])

    cleaned = re.sub(r'[\[\]"]', '', str(text))
    cleaned = cleaned.replace(".0", "")

    items = [item.strip() for item in cleaned.split(",")]

    emails = []
    phones = []

    for item in items:
        if "@" in item:
            emails.append(item)
        else:
            cleaned_phone = clean_phone(item)
            if cleaned_phone:
                phones.append(cleaned_phone)

    # Max 3 phones
    phone1 = phones[0] if len(phones) > 0 else ""
    phone2 = phones[1] if len(phones) > 1 else ""
    phone3 = phones[2] if len(phones) > 2 else ""

    return pd.Series([
        ", ".join(emails),
        phone1,
        phone2,
        phone3
    ])

# 3️⃣ Apply extraction (no row removed)
df[["Extracted_Emails", "Phone1", "Phone2", "Phone3"]] = df[raw_column_name].apply(extract_details)

# 4️⃣ Save output
output_file = "cleaned_output.xlsx"
df.to_excel(output_file, index=False)

# 5️⃣ Download
files.download(output_file)